# Milestone 2 — Exploratory Data Analysis
## IBM Telco Customer Churn Dataset

> **Purpose:** Understand the dataset structure, identify data-quality issues,
> explore feature distributions, and extract business insights that will guide
> feature engineering and model design in later milestones.
>
> **Status:** Under active development — Milestone 2 of 12.

---

## 0. Setup

In [ ]:
# ruff: noqa: E402
from __future__ import annotations

import sys
from pathlib import Path

# Ensure project root is on the path so src.* imports work
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

from src.data.loader import load_raw, raw_data_path
from src.data.splitting import split_data
from src.data.validation import validate_raw

# ── Plotting style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans',
})

CHURN_COLORS = {0: '#4878CF', 1: '#D65F5F'}  # blue=retained, red=churned
print('Setup complete. Project root:', PROJECT_ROOT)

---
## 1. Dataset Overview

In [ ]:
# Load raw data – no modifications
raw = load_raw()
print(f'Raw CSV path : {raw_data_path()}')
print(f'Shape        : {raw.shape[0]:,} rows × {raw.shape[1]} columns')

## 2. Shape and Column Inventory

In [ ]:
raw.head(3)

## 3. Data Types

In [ ]:
dtype_df = pd.DataFrame({'dtype': raw.dtypes, 'non_null': raw.notnull().sum(), 'sample': raw.iloc[0]})
print(dtype_df.to_string())

> **Note:** `TotalCharges` is object (string) — it contains 11 whitespace-only entries that must be converted to float.

## 4. Missing-Value Analysis

In [ ]:
# True NaN counts (pandas-detected)
nan_counts = raw.isnull().sum()
print('Columns with true NaN values:')
print(nan_counts[nan_counts > 0] if nan_counts.any() else 'None')

# Whitespace-only TotalCharges
whitespace_tc = (raw['TotalCharges'].str.strip() == '').sum()
print(f'\nTotalCharges whitespace-only rows: {whitespace_tc}')
print('These rows belong to customers with tenure=0 (brand-new subscribers).')
print(raw[raw['TotalCharges'].str.strip() == ''][['customerID','tenure','MonthlyCharges','TotalCharges']])

## 5. Duplicate Analysis

In [ ]:
n_dupes = raw.duplicated().sum()
n_id_dupes = raw['customerID'].duplicated().sum()
print(f'Fully duplicate rows  : {n_dupes}')
print(f'Duplicate customerIDs : {n_id_dupes}')
print('→ Dataset has no duplicate records and customerID is a unique identifier.')

---
## 6. Target Distribution (Churn)

In [ ]:
churn_counts = raw['Churn'].value_counts()
churn_pct = raw['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors = ['#4878CF', '#D65F5F']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=0.8)
for i, (_k, v) in enumerate(churn_counts.items()):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontsize=11)
axes[0].set_title('Churn Count')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Number of Customers')

# Pie chart
axes[1].pie(
    churn_counts.values,
    labels=churn_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
)
axes[1].set_title('Churn Proportion')

fig.suptitle('Target Distribution: Customer Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Retained (No)  : {churn_counts["No"]:,}  ({churn_pct["No"]:.1f}%)')
print(f'Churned  (Yes) : {churn_counts["Yes"]:,}  ({churn_pct["Yes"]:.1f}%)')
print(f'\nClass imbalance ratio: {churn_counts["No"]/churn_counts["Yes"]:.1f}:1 (retained:churned)')
print('→ Moderate class imbalance. Model evaluation must use metrics beyond accuracy (e.g., F1, AUC-ROC).')

---
## 7. Numerical Feature Distributions

In [ ]:
# Prepare a working copy with TotalCharges as float
work = raw.copy()
work['TotalCharges'] = pd.to_numeric(work['TotalCharges'], errors='coerce')

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, num_cols, strict=False):
    for churn_val, label, color in [(0, 'Retained', '#4878CF'), (1, 'Churned', '#D65F5F')]:
        mask = work['Churn'].map({'No': 0, 'Yes': 1}) == churn_val
        ax.hist(
            work.loc[mask, col].dropna(),
            bins=30, alpha=0.6, label=label,
            color=color, edgecolor='white', linewidth=0.4
        )
    ax.set_title(f'{col} Distribution by Churn')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()

fig.suptitle('Numerical Feature Distributions by Churn Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
print(work[num_cols].describe().round(2).to_string())

---
## 8. Categorical Feature Distributions

In [ ]:
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols, strict=False):
    counts = raw[col].value_counts()
    bars = ax.barh(counts.index, counts.values, color='#4878CF', edgecolor='white')
    for bar, v in zip(bars, counts.values, strict=False):
        ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
                f'{v:,}', va='center', fontsize=9)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Count')

fig.suptitle('Categorical Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Churn Rate by Contract Type

*Business question: Do contract lengths correlate with observed churn rates?*

In [ ]:
contract_churn = (
    raw.groupby('Contract')['Churn']
    .apply(lambda s: (s == 'Yes').mean() * 100)
    .sort_values(ascending=False)
)
counts = raw['Contract'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    contract_churn.index, contract_churn.values,
    color=['#D65F5F', '#F4A261', '#4878CF'],
    edgecolor='white'
)
for bar, (k, v) in zip(bars, contract_churn.items(), strict=False):
    n = counts[k]
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{v:.1f}%\n(n={n:,})', ha='center', fontsize=10)

ax.set_title('Observed Churn Rate by Contract Type', fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Contract Type')
ax.set_ylim(0, contract_churn.max() * 1.25)
plt.tight_layout()
plt.show()

print(contract_churn.round(1))

---
## 10. Churn Rate by Tenure

In [ ]:
work['Churn_int'] = work['Churn'].map({'No': 0, 'Yes': 1})
work['tenure_bin'] = pd.cut(work['tenure'], bins=[0,6,12,24,36,48,60,72,999],
                             labels=['0-6','7-12','13-24','25-36','37-48','49-60','61-72','>72'],
                             right=True, include_lowest=True)

tenure_churn = work.groupby('tenure_bin', observed=True).agg(
    churn_rate=('Churn_int', 'mean'),
    count=('Churn_int', 'count')
).reset_index()
tenure_churn['churn_pct'] = tenure_churn['churn_rate'] * 100

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.bar(tenure_churn['tenure_bin'].astype(str), tenure_churn['churn_pct'],
        color='#D65F5F', alpha=0.8, label='Churn Rate')
ax2.plot(tenure_churn['tenure_bin'].astype(str), tenure_churn['count'],
         color='#4878CF', marker='o', linewidth=2, label='Customer Count')

ax1.set_xlabel('Tenure (months)')
ax1.set_ylabel('Observed Churn Rate (%)', color='#D65F5F')
ax2.set_ylabel('Customer Count', color='#4878CF')
ax1.set_title('Observed Churn Rate and Customer Count by Tenure Group', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

print(tenure_churn[['tenure_bin','churn_pct','count']].to_string(index=False))

---
## 11. Churn Rate by Monthly Charges

In [ ]:
bins = [0, 35, 50, 65, 80, 95, 120]
labels = ['<35', '35-50', '50-65', '65-80', '80-95', '>95']
work['charge_bin'] = pd.cut(work['MonthlyCharges'], bins=bins, labels=labels, right=True)

charge_churn = work.groupby('charge_bin', observed=True).agg(
    churn_pct=('Churn_int', lambda x: x.mean() * 100),
    count=('Churn_int', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(charge_churn['charge_bin'].astype(str), charge_churn['churn_pct'],
              color='#F4A261', edgecolor='white')
for bar, (_, row) in zip(bars, charge_churn.iterrows(), strict=False):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row.churn_pct:.1f}%\n(n={row["count"]:,})', ha='center', fontsize=9)

ax.set_title('Observed Churn Rate by Monthly Charges Band', fontweight='bold')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Observed Churn Rate (%)')
ax.set_ylim(0, charge_churn['churn_pct'].max() * 1.3)
plt.tight_layout()
plt.show()

---
## 12. Churn Rate by Internet Service

In [ ]:
internet_churn = (
    work.groupby('InternetService')['Churn_int']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'churn_rate', 'count': 'n'})
)
internet_churn['churn_pct'] = internet_churn['churn_rate'] * 100
internet_churn = internet_churn.sort_values('churn_pct', ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#D65F5F', '#F4A261', '#4878CF']
bars = ax.bar(internet_churn.index, internet_churn['churn_pct'], color=colors, edgecolor='white')
for bar, (_k, row) in zip(bars, internet_churn.iterrows(), strict=False):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row.churn_pct:.1f}%\n(n={row.n:,})', ha='center', fontsize=10)

ax.set_title('Observed Churn Rate by Internet Service Type', fontweight='bold')
ax.set_xlabel('Internet Service')
ax.set_ylabel('Observed Churn Rate (%)')
ax.set_ylim(0, internet_churn['churn_pct'].max() * 1.25)
plt.tight_layout()
plt.show()
print(internet_churn[['churn_pct','n']].round(1))

---
## 13. Churn Rate by Payment Method

In [ ]:
pay_churn = (
    work.groupby('PaymentMethod')['Churn_int']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'churn_rate', 'count': 'n'})
)
pay_churn['churn_pct'] = pay_churn['churn_rate'] * 100
pay_churn = pay_churn.sort_values('churn_pct', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors_4 = ['#D65F5F', '#F4A261', '#4878CF', '#59A14F']
bars = ax.barh(pay_churn.index, pay_churn['churn_pct'], color=colors_4, edgecolor='white')
for bar, v in zip(bars, pay_churn['churn_pct'], strict=False):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=10)

ax.set_title('Observed Churn Rate by Payment Method', fontweight='bold')
ax.set_xlabel('Observed Churn Rate (%)')
ax.set_xlim(0, pay_churn['churn_pct'].max() * 1.2)
plt.tight_layout()
plt.show()
print(pay_churn[['churn_pct','n']].round(1))

---
## 14. Service-Related Churn Analysis

In [ ]:
service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

records = []
for col in service_cols:
    for val in work[col].unique():
        mask = work[col] == val
        rate = work.loc[mask, 'Churn_int'].mean() * 100
        n = mask.sum()
        records.append({'Feature': col, 'Value': val, 'Churn Rate (%)': rate, 'Count': n})

svc_df = pd.DataFrame(records).sort_values(['Feature', 'Value'])

# Plot: churn rate heatmap by service subscription
pivot = svc_df.pivot(index='Feature', columns='Value', values='Churn Rate (%)').fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=60)
plt.colorbar(im, ax=ax, label='Observed Churn Rate (%)')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if v > 0:
            ax.text(j, i, f'{v:.0f}%', ha='center', va='center', fontsize=8,
                    color='white' if v > 40 else 'black')

ax.set_title('Observed Churn Rate by Service Subscription Status', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 15. Correlation Analysis

In [ ]:
# Point-biserial-style correlation: numerical features vs binary Churn
from src.data.cleaning import clean_raw as _clean
from src.data.features import engineer_features as _eng

cleaned, _ = _clean(raw)
featured = _eng(cleaned)

num_feature_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
                    'total_services', 'estimated_annual_charges']

corr_data = {}
for col in num_feature_cols:
    if col in featured.columns:
        corr_data[col] = pd.to_numeric(featured[col], errors='coerce')

corr_df = pd.DataFrame(corr_data)
corr_df['Churn'] = featured['Churn']
corr_matrix = corr_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation matrix heatmap
cmap = plt.cm.RdBu_r
im = axes[0].imshow(corr_matrix.values, cmap=cmap, vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_xticks(range(len(corr_matrix.columns)))
axes[0].set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
axes[0].set_yticks(range(len(corr_matrix.index)))
axes[0].set_yticklabels(corr_matrix.index)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix.columns)):
        axes[0].text(j, i, f'{corr_matrix.values[i,j]:.2f}',
                     ha='center', va='center', fontsize=8)
axes[0].set_title('Feature Correlation Matrix', fontweight='bold')

# Bar: correlation with Churn
churn_corr = corr_matrix['Churn'].drop('Churn').sort_values()
colors_corr = ['#D65F5F' if v > 0 else '#4878CF' for v in churn_corr.values]
axes[1].barh(churn_corr.index, churn_corr.values, color=colors_corr, edgecolor='white')
axes[1].axvline(0, color='gray', linewidth=0.8)
axes[1].set_title('Correlation with Churn (binary)', fontweight='bold')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.show()

print('Correlation with Churn:')
print(churn_corr.round(3).to_string())

---
## 16. Data Validation Report

In [ ]:
report = validate_raw(raw)
print(report.summary())

---
## 17. Feature Engineering Preview

In [ ]:
print('Engineered features:')
for col in ['tenure_group','total_services','is_new_customer',
            'is_month_to_month','has_online_security','has_tech_support',
            'estimated_annual_charges']:
    if col in featured.columns:
        print(f'  {col}: {featured[col].dtype}  |  sample: {featured[col].iloc[0]}')

featured.head(3)

---
## 18. Train / Validation / Test Split Preview

In [ ]:
split = split_data(cleaned)
print(split.summary())

---
## Key Findings

### 1. Overall Churn Rate
- The dataset contains **7,043 customers**, of whom **1,869 (26.5%) churned** and **5,174 (73.5%) were retained**.
- This represents a moderate class imbalance (~2.77:1). Evaluation must go beyond accuracy; metrics such as **F1-score, Precision-Recall AUC, and ROC-AUC** are more informative.

### 2. Data-Quality Issues
- **11 rows** have whitespace-only `TotalCharges` values. These are not null in pandas but cannot be converted to float. Inspection shows all 11 have `tenure = 0` — they are brand-new customers who have not yet been charged.
  - **Cleaning decision:** impute `TotalCharges = 0.0` for these rows (deterministic, no statistical fitting).
- No other missing or duplicate records were identified.

### 3. Contract Type
- Customers on **Month-to-month** contracts show a **~42%** observed churn rate.
- **One-year** and **Two-year** contract customers show churn rates of approximately **11%** and **3%**, respectively.
- Contract type is likely one of the strongest predictors in the dataset. The `is_month_to_month` engineered feature captures this directly.

### 4. Tenure
- Customers in the **0–12 month** tenure band show the highest observed churn rates (often above 45%).
- Churn rates decline sharply and consistently with longer tenure.
- This confirms a **high-risk early-life** pattern: acquiring and then losing customers quickly before they can generate long-term value.

### 5. Monthly Charges
- Customers in higher charge bands (>$65/month) show higher observed churn rates.
- This may partly reflect the association between Fiber optic internet (a premium service) and higher churn.
- **Caution:** observed association only; higher charges may be a confounder for other contract or service attributes.

### 6. Internet Service
- **Fiber optic** customers show ~41% observed churn, compared to ~19% for DSL and ~7% for customers with no internet service.
- This is a notable pattern and may reflect pricing, competition, or service quality factors.

### 7. Payment Method
- Customers using **Electronic check** show a notably higher observed churn rate (~45%) compared to other payment methods (~15–18%).
- Electronic check may serve as a proxy for less engaged or financially strained customers.

### 8. Service Bundle Depth
- Customers without **Online Security** or **Tech Support** show higher churn rates than those who subscribe to these services.
- Customers who have subscribed to more services (higher `total_services` count) tend to show lower churn, consistent with a deeper engagement effect.

### 9. Leakage Warning
- All feature engineering was performed using only features available at prediction time (no target variable used). 
- All learned preprocessing (scaling, encoding) will be applied **after** the train/validation/test split in later milestones to prevent data leakage.

---
> *This notebook was produced as part of Milestone 2. All findings are observational and correlation-based. No causal claims are made.*